# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from os.path import join, exists
from tqdm.auto import tqdm

# Import Functions
sys.path.append("../../")
from src.configs.default_configs import fn_pred, fn_pred_perf
from src.configs.blood_config import data_name, batch_size, eval_batch_size, data_name_ood_in, data_name_ood_out, data_name_ood_diff

from src.misc import *
from src.file_manager.filepath import FilePath

from src.models.resnet.model import ResNet18
from src.models.resnet_rue.train import train_resnet_rue
from src.models.resnet_rue.predict import get_rue_predictions_resnet
from src.training.train import train_model_w_best_param

from src.file_manager.load_save_model import load_model
from src.evaluation.inference import get_all_predictions, split_test_set
from src.file_manager.load_save_df import save_pred_df

from src.models.resnet_rue.model import RueResNet18
from src.data_generator.blood import load_bloodmnist_data_dict
from src.data_generator.raabin import load_raabin_data_dict
from src.data_generator.bonemarrow import load_bonemarrow_data_dict
from src.data_generator.chest_mnist import load_chestmnist_data_dict
from src.data_processing.ood_dataset_preprocessing import process_dataset_for_ood, left_join_datasets
from src.data_processing.dataloader import get_pytorch_split_dict_image

fp_notebooks_folder = "./"
fp_project_folder = join(fp_notebooks_folder, "../", "../", "../")
fp_processed_data_folder = join(fp_project_folder, "Data", "isic_preprocessed")
fp_bcn20000_data_folder = join(fp_project_folder, "Data", "bcn20000")
fp_chestmnsit_data_folder = join(fp_project_folder, "Data", "chestmnist")

from cur_seed import seed
# seed = 2024
cur_model_name="tuned"
override=False

fp = FilePath(data_name=data_name, seed=seed)
fp_ood_in = FilePath(data_name=data_name_ood_in, seed=seed)
fp_ood_out = FilePath(data_name=data_name_ood_out, seed=seed)
fp_ood_diff = FilePath(data_name=data_name_ood_diff, seed=seed)

# Get Data

In [ ]:
print("Loading Data Dict")
data_dict = load_bloodmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
num_ori_test = len(data_dict["test_df"])

# Add ID Class OOD into the Test Set
print("Loading ID Class OOD Data Dict")
data_dict_ood_in = load_raabin_data_dict(fp_preprocessed=fp_ood_in.get_preprocessed_folder())
data_dict_ood_in = process_dataset_for_ood(data_dict, data_dict_ood_in, seed)

# ODD Class OOD
print("Loading OOD Class OOD Data Dict")
data_dict_ood_out = load_bonemarrow_data_dict(fp_preprocessed=fp_ood_out.get_preprocessed_folder())
data_dict_ood_out = process_dataset_for_ood(data_dict, data_dict_ood_out, seed)

# OOD Modality
print("Loading OOD Modality OOD Data Dict")
data_dict_ood_diff = load_chestmnist_data_dict(
    fp_preprocessed=fp_ood_diff.get_preprocessed_folder(), only_test=True, num_classes=8)
data_dict_ood_diff = process_dataset_for_ood(data_dict, data_dict_ood_diff, seed)

data_dict = left_join_datasets(data_dict, data_dict_ood_in)

# Training Param

In [ ]:
params = dict(
    ModelClass=RueResNet18,
    data_dict=data_dict,
    batch_size=batch_size,
    eval_batch_size=batch_size,
    train_model_func=train_resnet_rue,
    metric_to_monitor="rue mae",
    maximise=False,
    train_param_dict = dict(max_epochs=500, lr=0.001, weight_decay=0.001, patience=5, optimizer="adamw"),
    seed=seed,
    fp=fp,
    pytorch_split_dict_func=get_pytorch_split_dict_image
)

# Training 

In [ ]:
if not exists(join(fp.get_fp_model(RueResNet18, cur_model_name=cur_model_name))):
    resnet_model = load_model(fp=fp, ModelClass=ResNet18, cur_model_name=cur_model_name)
    # display_layer_indices(resnet_model)
    rue_best_param = {"resnet_model": resnet_model, "last_feat_extractor_layer_index": 0}
    rue_model = train_model_w_best_param(
        **params,
        best_param=rue_best_param,
        cur_model_name="tuned"
    )

# Prediction

In [ ]:
if not exists(join(fp.get_fp_df(RueResNet18, fn_pred))):
    rue_model = load_model(fp=fp, ModelClass=RueResNet18, cur_model_name=cur_model_name)
    pred_df = get_all_predictions(
        model=rue_model, 
        data_dict=data_dict, 
        batch_size=batch_size, 
        eval_batch_size=eval_batch_size,
        pred_func=get_rue_predictions_resnet,
        seed=seed,
        pytorch_split_dict_func=get_pytorch_split_dict_image
    ) 
    pred_df = split_test_set(
        pred_df, split_col="split", new_split_col="split_perf", 
        num_ori_test=num_ori_test, labels=["Test-Blood", "Test-Raabin"])
    save_pred_df(pred_df=pred_df, fp=fp, ModelClass=RueResNet18)

# OOD Prediction

In [ ]:
rue_model = load_model(fp=fp, ModelClass=RueResNet18, cur_model_name=cur_model_name)
ood_dicts = {
    "ood_in_raabin": data_dict_ood_in, 
    "ood_out_bonemarrow": data_dict_ood_out,
    "ood_chestmnist": data_dict_ood_diff}
for label, cur_ood_data_dict in tqdm(ood_dicts.items(), total=len(ood_dicts)):
    pred_df_ood = get_all_predictions(
        model=rue_model, 
        data_dict=cur_ood_data_dict, 
        batch_size=batch_size, 
        eval_batch_size=batch_size,
        pred_func=get_rue_predictions_resnet,
        seed=seed,
        pytorch_split_dict_func=get_pytorch_split_dict_image,
        )
    save_pred_df(pred_df=pred_df_ood, fp=fp, ModelClass=RueResNet18, optional_label=label)